# 01 — Dataset Exploration & Data Integrity Audit

## Image Captioning Project — MS COCO 2017

### 1. Introduction

This notebook presents the dataset exploration and data integrity audit conducted for the image captioning project using the MS COCO 2017 dataset.

The main objective is to examine the dataset, verify the selected images, investigate annotation consistency, identify potential data quality issues, and generate audit reports.

### 2. Audit Objectives

* Explore the structure and organization of the MS COCO dataset.
* Examine image and annotation statistics.
* Verify the Knowledge Base and validation dataset selection.
* Identify missing, corrupted, and duplicate images.
* Check the consistency between images and their annotations.
* Investigate images without object annotations.
* Generate structured reports documenting the audit results.

### 3. Dataset Configuration

| Component             | Description     |
| --------------------- | --------------- |
| Dataset               | MS COCO 2017    |
| Image source          | train2017       |
| Knowledge Base        | 10,000 images   |
| Validation            | 100 images      |
| Total selected images | 10,100          |
| Selection method      | Random sampling |
| Random seed           | 42              |


In [ ]:
# ==========================================
# 1. Environment Setup
# ==========================================

import os
import json
import random
import hashlib
import pandas as pd

from collections import Counter, defaultdict
from PIL import Image
from tqdm.notebook import tqdm

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

print("Environment initialized successfully.")

Mounted at /content/drive
Environment initialized successfully.


In [ ]:
# ==========================================
# 2. Project Paths
# ==========================================

BASE_DIR = "/content/drive/MyDrive/COCO_Project"

ANNOTATIONS_DIR = os.path.join(BASE_DIR, "annotations")
METADATA_DIR = os.path.join(BASE_DIR, "metadata")

IMAGES_DIR = os.path.join(BASE_DIR, "images")

KB_DIR = os.path.join(IMAGES_DIR, "knowledge_base")
VAL_DIR = os.path.join(IMAGES_DIR, "validation")
TEST_DIR = os.path.join(IMAGES_DIR, "test")

# Test annotations from COCO val2017
VAL2017_ANNOTATIONS_PATH = os.path.join(
    ANNOTATIONS_DIR,
    "instances_val2017.json"
)

# Verify existing project folders
required_paths = [
    ANNOTATIONS_DIR,
    METADATA_DIR,
    KB_DIR,
    VAL_DIR
]

for path in required_paths:
    assert os.path.exists(path), f"Missing folder: {path}"

# Create test folder if it does not exist
os.makedirs(TEST_DIR, exist_ok=True)

print("All project folders verified successfully!")
print("Test folder:", TEST_DIR)

All project folders verified successfully!
Test folder: /content/drive/MyDrive/COCO_Project/images/test


In [ ]:
# ==========================================
# 3. Load COCO val2017 Annotations
# ==========================================

assert os.path.exists(VAL2017_ANNOTATIONS_PATH), (
    f"Missing file: {VAL2017_ANNOTATIONS_PATH}"
)

with open(VAL2017_ANNOTATIONS_PATH, "r") as f:
    val2017_instances = json.load(f)

print("COCO val2017 annotations loaded successfully!")

print("Number of images:", len(val2017_instances["images"]))
print("Number of categories:", len(val2017_instances["categories"]))
print("Number of annotations:", len(val2017_instances["annotations"]))

COCO val2017 annotations loaded successfully!
Number of images: 5000
Number of categories: 80
Number of annotations: 36781


In [ ]:
# Check available annotation files

print("Files inside annotations folder:\n")

for file in os.listdir(ANNOTATIONS_DIR):
    print(file)

Files inside annotations folder:

instances_train2017.json
captions_train2017.json
captions_val2017.json
captions_test100.json
instances_val2017.json


In [ ]:
import urllib.request
import zipfile

# Download COCO 2017 validation annotations
url = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"

zip_path = os.path.join(BASE_DIR, "annotations_trainval2017.zip")

urllib.request.urlretrieve(url, zip_path)

print("Annotations archive downloaded successfully!")

Annotations archive downloaded successfully!


In [ ]:
# Extract only instances_val2017.json

with zipfile.ZipFile(zip_path, "r") as zip_ref:

    target_file = "annotations/instances_val2017.json"

    assert target_file in zip_ref.namelist(), (
        "instances_val2017.json was not found in the archive."
    )

    zip_ref.extract(target_file, BASE_DIR)

print("instances_val2017.json extracted successfully!")

instances_val2017.json extracted successfully!


In [ ]:
# ==========================================
# 3. Load COCO Annotations
# ==========================================

# Train2017 annotation paths
instances_path = os.path.join(
    ANNOTATIONS_DIR,
    "instances_train2017.json"
)

captions_path = os.path.join(
    ANNOTATIONS_DIR,
    "captions_train2017.json"
)

# Val2017 annotation path
val2017_instances_path = os.path.join(
    ANNOTATIONS_DIR,
    "instances_val2017.json"
)

# Verify annotation files
assert os.path.exists(instances_path), "Train instances JSON not found."
assert os.path.exists(captions_path), "Train captions JSON not found."
assert os.path.exists(val2017_instances_path), "Val2017 instances JSON not found."

# Load Train2017 JSON files
with open(instances_path, "r") as f:
    instances = json.load(f)

with open(captions_path, "r") as f:
    captions = json.load(f)

# Load Val2017 instances
with open(val2017_instances_path, "r") as f:
    val2017_instances = json.load(f)

print("All annotation files loaded successfully!")

print("\nTrain2017:")
print("Images:", len(instances["images"]))
print("Categories:", len(instances["categories"]))
print("Annotations:", len(instances["annotations"]))

print("\nVal2017:")
print("Images:", len(val2017_instances["images"]))
print("Categories:", len(val2017_instances["categories"]))
print("Annotations:", len(val2017_instances["annotations"]))

All annotation files loaded successfully!

Train2017:
Images: 118287
Categories: 80
Annotations: 860001

Val2017:
Images: 5000
Categories: 80
Annotations: 36781


In [ ]:
# ==========================================
# 4. Check Existing Test Images
# ==========================================

existing_test_images = [
    f for f in os.listdir(TEST_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print("Existing test images:", len(existing_test_images))

Existing test images: 1000


In [ ]:
# ==========================================
# 5. Select 1,000 Test Images
# ==========================================

# Make selection reproducible
random.seed(42)

# Get all val2017 image IDs
val2017_image_ids = [
    image["id"] for image in val2017_instances["images"]
]

# Select 1,000 unique images
test_image_ids = random.sample(val2017_image_ids, 1000)

# Verify selection
assert len(test_image_ids) == 1000
assert len(set(test_image_ids)) == 1000

print("Test image selection completed!")
print("Total val2017 images:", len(val2017_image_ids))
print("Selected test images:", len(test_image_ids))
print("Unique test IDs:", len(set(test_image_ids)))

Test image selection completed!
Total val2017 images: 5000
Selected test images: 1000
Unique test IDs: 1000


In [ ]:
# ==========================================
# 6. Prepare Test Image Metadata
# ==========================================

# Create a lookup dictionary
val2017_image_info = {
    image["id"]: image
    for image in val2017_instances["images"]
}

# Retrieve metadata for selected test images
test_image_metadata = [
    val2017_image_info[image_id]
    for image_id in test_image_ids
]

# Create a DataFrame
test_df = pd.DataFrame(test_image_metadata)

# Keep the essential columns
test_df = test_df[
    ["id", "file_name", "coco_url", "width", "height"]
]

print("Test metadata prepared successfully!")
print("Number of selected images:", len(test_df))

display(test_df.head())

Test metadata prepared successfully!
Number of selected images: 1000


,id,file_name,coco_url,width,height
0,301061,000000301061.jpg,http://images.cocodataset.org/val2017/00000030...,480,640
1,261982,000000261982.jpg,http://images.cocodataset.org/val2017/00000026...,457,640
2,273760,000000273760.jpg,http://images.cocodataset.org/val2017/00000027...,467,640
3,248400,000000248400.jpg,http://images.cocodataset.org/val2017/00000024...,640,427
4,382030,000000382030.jpg,http://images.cocodataset.org/val2017/00000038...,500,375


In [ ]:
# ==========================================
# 7. Save Test Image Selection
# ==========================================

test_metadata_path = os.path.join(
    METADATA_DIR,
    "test_1000_metadata.csv"
)

# Save metadata
test_df.to_csv(test_metadata_path, index=False)

# Save selected image IDs
test_ids_path = os.path.join(
    METADATA_DIR,
    "test_1000_image_ids.csv"
)

pd.DataFrame({
    "image_id": test_image_ids
}).to_csv(test_ids_path, index=False)

# Verify saved files
assert os.path.exists(test_metadata_path)
assert os.path.exists(test_ids_path)

print("Test selection saved successfully!")
print("Metadata file:", test_metadata_path)
print("Image IDs file:", test_ids_path)
print("Saved images:", len(test_df))

Test selection saved successfully!
Metadata file: /content/drive/MyDrive/COCO_Project/metadata/test_1000_metadata.csv
Image IDs file: /content/drive/MyDrive/COCO_Project/metadata/test_1000_image_ids.csv
Saved images: 1000


In [ ]:
# ==========================================
# 8. Check Previous 100-Image Test Version
# ==========================================

old_test100_path = os.path.join(
    ANNOTATIONS_DIR,
    "captions_test100.json"
)

assert os.path.exists(old_test100_path), (
    "Previous 100-image test annotations not found."
)

with open(old_test100_path, "r") as f:
    old_test100 = json.load(f)

# Extract image IDs
old_test100_ids = {
    image["id"] for image in old_test100["images"]
}

# Compare old and new test selections
overlap_ids = set(test_image_ids).intersection(old_test100_ids)

print("Previous test images:", len(old_test100_ids))
print("New test images:", len(test_image_ids))
print("Overlapping images:", len(overlap_ids))
print("New images not in previous test:",
      len(set(test_image_ids) - old_test100_ids))

Previous test images: 100
New test images: 1000
Overlapping images: 100
New images not in previous test: 900


In [ ]:
# ==========================================
# 9. Download Test Images
# ==========================================

import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

def download_image(row):
    filename = row["file_name"]
    url = row["coco_url"]

    save_path = os.path.join(TEST_DIR, filename)

    # Skip already downloaded images
    if os.path.exists(save_path):
        return "Already exists"

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        with open(save_path, "wb") as f:
            f.write(response.content)

        return "Downloaded"

    except Exception as e:
        return f"Failed: {e}"


results = []

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [
        executor.submit(download_image, row)
        for _, row in test_df.iterrows()
    ]

    for future in tqdm(as_completed(futures), total=len(futures)):
        results.append(future.result())


print("\nDownload completed!")
print("Downloaded:", results.count("Downloaded"))
print("Already existed:", results.count("Already exists"))
print("Failed:", sum(r.startswith("Failed") for r in results))

  0%|          | 0/1000 [00:00<?, ?it/s]


Download completed!
Downloaded: 0
Already existed: 1000
Failed: 0


In [ ]:
# ==========================================
# 10. Verify Test Image Integrity
# ==========================================

valid_images = []
corrupted_images = []

for filename in tqdm(test_df["file_name"]):

    image_path = os.path.join(TEST_DIR, filename)

    try:
        with Image.open(image_path) as img:
            img.verify()

        valid_images.append(filename)

    except Exception:
        corrupted_images.append(filename)

print("\nTest Image Integrity Report")
print("---------------------------")
print("Expected images:", len(test_df))
print("Valid images:", len(valid_images))
print("Corrupted images:", len(corrupted_images))

assert len(valid_images) == 1000, "Some test images are invalid!"
assert len(corrupted_images) == 0, "Corrupted images detected!"

print("\nAll 1,000 test images passed the integrity check!")

  0%|          | 0/1000 [00:00<?, ?it/s]


Test Image Integrity Report
---------------------------
Expected images: 1000
Valid images: 1000
Corrupted images: 0

All 1,000 test images passed the integrity check!


In [ ]:
# ==========================================
# 11. Check Dataset Split Independence
# ==========================================

# Load existing training and validation selections
selected_ids_path = os.path.join(
    METADATA_DIR,
    "selected_image_ids.csv"
)

assert os.path.exists(selected_ids_path), (
    "selected_image_ids.csv not found."
)

selected_df = pd.read_csv(selected_ids_path)

print("Available columns:", selected_df.columns.tolist())
print(selected_df.head())

Available columns: ['image_id', 'split', 'file_name']
   image_id           split         file_name
0    411685  knowledge_base  000000411685.jpg
1     71573  knowledge_base  000000071573.jpg
2     16285  knowledge_base  000000016285.jpg
3    478093  knowledge_base  000000478093.jpg
4    177361  knowledge_base  000000177361.jpg


In [ ]:
# ==========================================
# 12. Verify Dataset Split Independence
# ==========================================

# Separate Knowledge Base and validation IDs
kb_ids = set(
    selected_df[selected_df["split"] == "knowledge_base"]["image_id"]
)

validation_ids = set(
    selected_df[selected_df["split"] == "validation"]["image_id"]
)

test_ids = set(test_image_ids)

# Check overlaps
kb_validation_overlap = kb_ids.intersection(validation_ids)
kb_test_overlap = kb_ids.intersection(test_ids)
validation_test_overlap = validation_ids.intersection(test_ids)

# Display results
print("DATASET SPLIT VERIFICATION")
print("--------------------------")

print("Knowledge Base images:", len(kb_ids))
print("Validation images:", len(validation_ids))
print("Test images:", len(test_ids))

print("\nOVERLAP CHECKS")
print("--------------------------")

print("Knowledge Base / Validation:", len(kb_validation_overlap))
print("Knowledge Base / Test:", len(kb_test_overlap))
print("Validation / Test:", len(validation_test_overlap))

# Assertions
assert len(kb_ids) == 10000
assert len(validation_ids) == 100
assert len(test_ids) == 1000

assert len(kb_validation_overlap) == 0
assert len(kb_test_overlap) == 0
assert len(validation_test_overlap) == 0

print("\nSUCCESS: All three datasets are independent!")

DATASET SPLIT VERIFICATION
--------------------------
Knowledge Base images: 10000
Validation images: 100
Test images: 1000

OVERLAP CHECKS
--------------------------
Knowledge Base / Validation: 0
Knowledge Base / Test: 0
Validation / Test: 0

SUCCESS: All three datasets are independent!


In [ ]:
# ==========================================
# 13. Create Unified Dataset Manifest
# ==========================================

# Prepare Knowledge Base and validation metadata
train_metadata = selected_df.copy()

train_metadata["source"] = "train2017"

# Prepare test metadata
test_manifest = test_df[["id", "file_name"]].copy()

test_manifest = test_manifest.rename(
    columns={"id": "image_id"}
)

test_manifest["split"] = "test"
test_manifest["source"] = "val2017"

# Combine all datasets
dataset_manifest = pd.concat(
    [
        train_metadata[["image_id", "split", "file_name", "source"]],
        test_manifest[["image_id", "split", "file_name", "source"]]
    ],
    ignore_index=True
)

# Verify manifest
assert len(dataset_manifest) == 11100
assert dataset_manifest["image_id"].is_unique

# Save manifest
manifest_path = os.path.join(
    METADATA_DIR,
    "dataset_manifest.csv"
)

dataset_manifest.to_csv(manifest_path, index=False)

print("Unified dataset manifest created successfully!")
print("\nDataset distribution:")
print(dataset_manifest["split"].value_counts())

print("\nTotal images:", len(dataset_manifest))
print("Unique image IDs:", dataset_manifest["image_id"].nunique())
print("Saved to:", manifest_path)

Unified dataset manifest created successfully!

Dataset distribution:
split
knowledge_base    10000
test               1000
validation          100
Name: count, dtype: int64

Total images: 11100
Unique image IDs: 11100
Saved to: /content/drive/MyDrive/COCO_Project/metadata/dataset_manifest.csv


In [ ]:
# ==========================================
# 14. Verify All Dataset Image Files
# ==========================================

split_directories = {
    "knowledge_base": KB_DIR,
    "validation": VAL_DIR,
    "test": TEST_DIR
}

missing_files = []

for _, row in tqdm(dataset_manifest.iterrows(),
                   total=len(dataset_manifest)):

    image_dir = split_directories[row["split"]]
    image_path = os.path.join(image_dir, row["file_name"])

    if not os.path.exists(image_path):
        missing_files.append(row["file_name"])

print("\nIMAGE FILE VERIFICATION")
print("-----------------------")
print("Expected images:", len(dataset_manifest))
print("Missing images:", len(missing_files))
print("Existing images:", len(dataset_manifest) - len(missing_files))

if missing_files:
    print("\nFirst missing files:")
    print(missing_files[:20])
else:
    print("\nSUCCESS: All 11,100 image files exist!")

  0%|          | 0/11100 [00:00<?, ?it/s]


IMAGE FILE VERIFICATION
-----------------------
Expected images: 11100
Missing images: 0
Existing images: 11100

SUCCESS: All 11,100 image files exist!


In [ ]:
# ==========================================
# 4. Dataset Overview
# ==========================================

print("========== INSTANCES DATASET ==========")

print("Number of images:", len(instances["images"]))
print("Number of object annotations:", len(instances["annotations"]))
print("Number of categories:", len(instances["categories"]))

print("\n========== CAPTIONS DATASET ==========")

print("Number of images:", len(captions["images"]))
print("Number of caption annotations:", len(captions["annotations"]))

========== INSTANCES DATASET ==========
Number of images: 118287
Number of object annotations: 860001
Number of categories: 80

========== CAPTIONS DATASET ==========
Number of images: 118287
Number of caption annotations: 591753


In [ ]:
# ==========================================
# 15. Detect Duplicate Images Across Splits
# ==========================================

import hashlib
from collections import defaultdict

hash_to_files = defaultdict(list)

for _, row in tqdm(
    dataset_manifest.iterrows(),
    total=len(dataset_manifest)
):

    image_dir = split_directories[row["split"]]
    image_path = os.path.join(image_dir, row["file_name"])

    sha256_hash = hashlib.sha256()

    with open(image_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256_hash.update(chunk)

    hash_to_files[sha256_hash.hexdigest()].append({
        "image_id": row["image_id"],
        "split": row["split"],
        "file_name": row["file_name"]
    })

# Identify duplicate files
duplicate_groups = {
    h: files
    for h, files in hash_to_files.items()
    if len(files) > 1
}

print("\nSHA-256 DUPLICATE REPORT")
print("------------------------")
print("Total images checked:", len(dataset_manifest))
print("Unique hashes:", len(hash_to_files))
print("Duplicate groups:", len(duplicate_groups))

# Save duplicate report
duplicate_records = []

for files in duplicate_groups.values():
    duplicate_records.extend(files)

duplicate_df = pd.DataFrame(duplicate_records)

duplicate_path = os.path.join(
    METADATA_DIR,
    "cross_split_duplicates.csv"
)

duplicate_df.to_csv(duplicate_path, index=False)

print("Duplicate report saved:", duplicate_path)

  0%|          | 0/11100 [00:00<?, ?it/s]


SHA-256 DUPLICATE REPORT
------------------------
Total images checked: 11100
Unique hashes: 11100
Duplicate groups: 0
Duplicate report saved: /content/drive/MyDrive/COCO_Project/metadata/cross_split_duplicates.csv


In [ ]:
# ==========================================
# 5. Check Image ID Consistency
# ==========================================

# Extract image IDs from both annotation files
instance_ids = {
    image["id"] for image in instances["images"]
}

caption_ids = {
    image["id"] for image in captions["images"]
}

# Find common and different IDs
common_ids = instance_ids.intersection(caption_ids)

missing_from_captions = instance_ids - caption_ids
missing_from_instances = caption_ids - instance_ids

# Display results
print("========== IMAGE ID CONSISTENCY ==========")

print("Images in instances JSON:", len(instance_ids))
print("Images in captions JSON:", len(caption_ids))
print("Images common to both:", len(common_ids))

print("\nMissing from captions JSON:", len(missing_from_captions))
print("Missing from instances JSON:", len(missing_from_instances))

print("\nConsistency:",
      "PASSED" if instance_ids == caption_ids else "FAILED")

========== IMAGE ID CONSISTENCY ==========
Images in instances JSON: 118287
Images in captions JSON: 118287
Images common to both: 118287

Missing from captions JSON: 0
Missing from instances JSON: 0

Consistency: PASSED


In [ ]:
# ==========================================
# 6. Load Selected Dataset IDs
# ==========================================

selection_path = os.path.join(
    METADATA_DIR,
    "selected_image_ids.csv"
)

# Verify that the selection file exists
assert os.path.exists(selection_path), "Selection CSV not found!"

# Load the saved selection
selection_df = pd.read_csv(selection_path)

# Separate Knowledge Base and validation IDs
kb_df = selection_df[
    selection_df["split"] == "knowledge_base"
]

val_df = selection_df[
    selection_df["split"] == "validation"
]

knowledge_base_ids = kb_df["image_id"].tolist()
validation_ids = val_df["image_id"].tolist()

print("========== SELECTED DATASET ==========")

print("Total selected images:", len(selection_df))
print("Knowledge Base images:", len(knowledge_base_ids))
print("Validation images:", len(validation_ids))

========== SELECTED DATASET ==========
Total selected images: 10100
Knowledge Base images: 10000
Validation images: 100


## 7. Verify Dataset Selection

This section verifies that the selected dataset contains exactly 10,000 Knowledge Base images and 100 validation images, with no duplicated IDs or overlap between the two subsets.


In [ ]:
# ==========================================
# 7. Verify Dataset Selection
# ==========================================

# Check expected dataset sizes
assert len(knowledge_base_ids) == 10000, "Incorrect Knowledge Base size!"
assert len(validation_ids) == 100, "Incorrect validation size!"

# Check for duplicate IDs within each subset
assert len(knowledge_base_ids) == len(set(knowledge_base_ids)), \
    "Duplicate IDs found in Knowledge Base!"

assert len(validation_ids) == len(set(validation_ids)), \
    "Duplicate IDs found in validation!"

# Check for overlap between the two subsets
overlap_ids = set(knowledge_base_ids).intersection(set(validation_ids))

print("========== DATASET SELECTION VERIFICATION ==========")

print("Knowledge Base size:", len(knowledge_base_ids))
print("Validation size:", len(validation_ids))
print("Total selected images:", len(knowledge_base_ids) + len(validation_ids))
print("Duplicate IDs in Knowledge Base:",
      len(knowledge_base_ids) - len(set(knowledge_base_ids)))
print("Duplicate IDs in validation:",
      len(validation_ids) - len(set(validation_ids)))
print("Overlap between subsets:", len(overlap_ids))

print("\nSelection verification: PASSED")

========== DATASET SELECTION VERIFICATION ==========
Knowledge Base size: 10000
Validation size: 100
Total selected images: 10100
Duplicate IDs in Knowledge Base: 0
Duplicate IDs in validation: 0
Overlap between subsets: 0

Selection verification: PASSED


In [ ]:
import os
import json
import random
import hashlib
import pandas as pd

from collections import Counter, defaultdict
from PIL import Image
from tqdm.notebook import tqdm

from google.colab import drive

drive.mount('/content/drive')

print("Environment initialized successfully.")

In [ ]:
# ==========================================
# 8. Verify Image Files
# ==========================================

def get_image_files(folder):
    return {
        os.path.splitext(filename)[0]
        for filename in os.listdir(folder)
        if filename.lower().endswith((".jpg", ".jpeg", ".png"))
    }

kb_files = get_image_files(KB_DIR)
val_files = get_image_files(VAL_DIR)

print("========== IMAGE FILE VERIFICATION ==========")

print("Knowledge Base image files:", len(kb_files))
print("Validation image files:", len(val_files))

print("Expected Knowledge Base images:", len(knowledge_base_ids))
print("Expected validation images:", len(validation_ids))

In [ ]:
# ==========================================
# 9. Check for Missing Images
# ==========================================

# Format image IDs as 12-digit filenames
expected_kb_files = {
    f"{image_id:012d}" for image_id in knowledge_base_ids
}

expected_val_files = {
    f"{image_id:012d}" for image_id in validation_ids
}

# Identify missing files
missing_kb_images = expected_kb_files - kb_files
missing_val_images = expected_val_files - val_files

print("========== MISSING IMAGE CHECK ==========")

print("Missing Knowledge Base images:", len(missing_kb_images))
print("Missing validation images:", len(missing_val_images))

print("Total missing images:",
      len(missing_kb_images) + len(missing_val_images))

In [ ]:
print("Knowledge Base filenames:")
print(os.listdir(KB_DIR)[:10])

print("\nValidation filenames:")
print(os.listdir(VAL_DIR)[:10])

In [ ]:
# ==========================================
# 10. Check for Corrupted Images
# ==========================================

corrupted_images = []

all_image_paths = [
    (os.path.join(KB_DIR, filename), "knowledge_base")
    for filename in os.listdir(KB_DIR)
    if filename.lower().endswith((".jpg", ".jpeg", ".png"))
]

all_image_paths += [
    (os.path.join(VAL_DIR, filename), "validation")
    for filename in os.listdir(VAL_DIR)
    if filename.lower().endswith((".jpg", ".jpeg", ".png"))
]

for image_path, split in tqdm(all_image_paths, desc="Checking images"):
    try:
        with Image.open(image_path) as img:
            img.verify()
    except Exception:
        corrupted_images.append({
            "filename": os.path.basename(image_path),
            "split": split
        })

print("========== IMAGE INTEGRITY CHECK ==========")
print("Total images checked:", len(all_image_paths))
print("Corrupted images:", len(corrupted_images))
print("Valid images:", len(all_image_paths) - len(corrupted_images))

In [ ]:
# ==========================================
# 11. Detect Exact Duplicate Images
# ==========================================

hash_to_images = defaultdict(list)

for image_path, split in tqdm(all_image_paths, desc="Detecting duplicates"):
    with open(image_path, "rb") as file:
        image_hash = hashlib.sha256(file.read()).hexdigest()

    hash_to_images[image_hash].append({
        "filename": os.path.basename(image_path),
        "split": split
    })

# Keep only hashes associated with multiple files
duplicate_images = [
    {
        "sha256": image_hash,
        "files": files
    }
    for image_hash, files in hash_to_images.items()
    if len(files) > 1
]

print("========== EXACT DUPLICATE CHECK ==========")
print("Total images hashed:", len(all_image_paths))
print("Duplicate groups:", len(duplicate_images))

In [ ]:
# ==========================================
# 12. Verify Selected Images Against Annotations
# ==========================================

selected_ids = set(knowledge_base_ids + validation_ids)

# Extract IDs from annotations
instance_ids = {image["id"] for image in instances["images"]}
caption_ids = {image["id"] for image in captions["images"]}

# Check selected IDs
missing_from_instances = selected_ids - instance_ids
missing_from_captions = selected_ids - caption_ids

print("========== ANNOTATION CONSISTENCY CHECK ==========")

print("Total selected images:", len(selected_ids))
print("Missing from instances annotations:", len(missing_from_instances))
print("Missing from captions annotations:", len(missing_from_captions))

print(
    "\nAnnotation consistency:",
    "PASSED" if not missing_from_instances and not missing_from_captions else "FAILED"
)

In [ ]:
# ==========================================
# 13. Identify Images Without Object Annotations
# ==========================================

# Count object annotations for each selected image
object_counts = Counter(
    annotation["image_id"]
    for annotation in instances["annotations"]
    if annotation["image_id"] in selected_ids
)

# Identify selected images with zero object annotations
images_without_objects = [
    image_id
    for image_id in selected_ids
    if object_counts[image_id] == 0
]

# Count captions for each selected image
caption_counts = Counter(
    annotation["image_id"]
    for annotation in captions["annotations"]
    if annotation["image_id"] in selected_ids
)

print("========== OBJECT ANNOTATION CHECK ==========")
print("Selected images:", len(selected_ids))
print("Images without object annotations:", len(images_without_objects))
print("Images with object annotations:", len(selected_ids) - len(images_without_objects))
print("Images without captions:", sum(
    1 for image_id in selected_ids if caption_counts[image_id] == 0
))

In [ ]:
# ==========================================
# 14. Investigate Images Without Object Annotations
# ==========================================

# Count captions for the 104 images
no_object_caption_counts = {
    image_id: caption_counts[image_id]
    for image_id in images_without_objects
}

# Summarize caption counts
caption_distribution = Counter(no_object_caption_counts.values())

print("========== NO-OBJECT IMAGES INVESTIGATION ==========")

print("Images investigated:", len(images_without_objects))
print("Caption count distribution:")

for count, frequency in sorted(caption_distribution.items()):
    print(f"{count} caption(s): {frequency} image(s)")

print("\nImages without captions:",
      sum(1 for count in no_object_caption_counts.values() if count == 0))

In [ ]:
# ==========================================
# 15. Save Corrupted Images Report
# ==========================================

corrupted_df = pd.DataFrame(
    corrupted_images,
    columns=["filename", "split"]
)

corrupted_path = os.path.join(
    METADATA_DIR,
    "corrupted_images.csv"
)

corrupted_df.to_csv(corrupted_path, index=False)

print("========== CORRUPTED IMAGES REPORT ==========")
print("File saved:", corrupted_path)
print("Number of corrupted images:", len(corrupted_df))
print("Report saved successfully!")

In [ ]:
# ==========================================
# 16. Save Duplicate Images Report
# ==========================================

duplicate_rows = []

for group in duplicate_images:
    for file_info in group["files"]:
        duplicate_rows.append({
            "sha256": group["sha256"],
            "filename": file_info["filename"],
            "split": file_info["split"]
        })

duplicate_df = pd.DataFrame(
    duplicate_rows,
    columns=["sha256", "filename", "split"]
)

duplicate_path = os.path.join(
    METADATA_DIR,
    "duplicate_images.csv"
)

duplicate_df.to_csv(duplicate_path, index=False)

print("========== DUPLICATE IMAGES REPORT ==========")
print("File saved:", duplicate_path)
print("Number of duplicate groups:", len(duplicate_images))
print("Report saved successfully!")

In [ ]:
# ==========================================
# 17. Save Images Without Object Annotations
# ==========================================

no_objects_df = pd.DataFrame({
    "image_id": images_without_objects,
    "caption_count": [
        caption_counts[image_id]
        for image_id in images_without_objects
    ]
})

no_objects_path = os.path.join(
    METADATA_DIR,
    "images_without_object_annotations.csv"
)

no_objects_df.to_csv(no_objects_path, index=False)

print("========== NO-OBJECT IMAGES REPORT ==========")
print("File saved:", no_objects_path)
print("Images without object annotations:", len(no_objects_df))
print("Report saved successfully!")

In [ ]:

# ==========================================
# 18. Generate Updated Dataset Audit Report
# ==========================================

audit_report = {
    "dataset": "MS COCO 2017",

    "dataset_splits": {
        "knowledge_base": {
            "source": "train2017",
            "images": len(kb_ids)
        },
        "validation": {
            "source": "train2017",
            "images": len(validation_ids)
        },
        "test": {
            "source": "val2017",
            "images": len(test_ids)
        },
        "total_selected_images": len(dataset_manifest)
    },

    "selection": {
        "random_seed": 42,
        "knowledge_base_images": len(kb_ids),
        "validation_images": len(validation_ids),
        "test_images": len(test_ids),
        "unique_image_ids": dataset_manifest["image_id"].nunique(),
        "kb_validation_overlap": len(kb_validation_overlap),
        "kb_test_overlap": len(kb_test_overlap),
        "validation_test_overlap": len(validation_test_overlap)
    },

    "previous_test_comparison": {
        "previous_test_images": len(old_test100_ids),
        "new_test_images": len(test_ids),
        "overlapping_images": len(overlap_ids),
        "additional_images": len(test_ids - old_test100_ids)
    },

    "source_annotations": {
        "train2017_images_in_instances": len(instances["images"]),
        "train2017_images_in_captions": len(captions["images"]),
        "train2017_object_annotations": len(instances["annotations"]),
        "train2017_caption_annotations": len(captions["annotations"]),
        "object_categories": len(instances["categories"]),
        "image_id_consistency": instance_ids == caption_ids
    },

    "val2017_annotations": {
        "images": len(val2017_instances["images"]),
        "object_annotations": len(val2017_instances["annotations"]),
        "categories": len(val2017_instances["categories"])
    },

    "image_integrity": {
        "total_images_checked": len(dataset_manifest),
        "missing_images": len(missing_files),
        "corrupted_test_images": len(corrupted_images),
        "unique_sha256_hashes": 11100,
        "exact_duplicate_groups": len(duplicate_groups)
    },

    "annotation_consistency": {
        "selected_images_missing_from_instances": len(missing_from_instances),
        "selected_images_missing_from_captions": len(missing_from_captions),
        "images_without_object_annotations": len(images_without_objects),
        "images_without_captions": sum(
            1 for image_id in selected_ids
            if caption_counts[image_id] == 0
        )
    },

    "no_object_images_investigation": {
        "images_investigated": len(images_without_objects),
        "caption_count_distribution": {
    str(count): frequency
    for count, frequency in sorted(
        Counter(caption_counts[image_id] for image_id in images_without_objects).items()
    )
}
    },

    "output_files": {
        "dataset_manifest": "dataset_manifest.csv",
        "test_metadata": "test_1000_metadata.csv",
        "test_image_ids": "test_1000_image_ids.csv",
        "cross_split_duplicates": "cross_split_duplicates.csv"
    },

    "notes": [
        "Knowledge Base contains 10,000 images from COCO train2017.",
        "Validation contains 100 different images from COCO train2017.",
        "Test contains 1,000 images from COCO val2017.",
        "The previous 100-image test version is preserved for comparison.",
        "All three dataset splits were verified to be independent.",
        "SHA-256 hashing was used to detect exact duplicate images.",
        "Images without object annotations were investigated using their captions."
    ]
}

# Save updated audit report
audit_report_path = os.path.join(
    METADATA_DIR,
    "dataset_audit_report.json"
)

with open(audit_report_path, "w") as file:
    json.dump(audit_report, file, indent=4)

print("========== UPDATED DATASET AUDIT REPORT ==========")
print("Report saved:", audit_report_path)
print("Report generated successfully!")


In [ ]:
# ==========================================
# 19. Document Dataset Structure
# ==========================================

structure_text = """
MS COCO 2017 — IMAGE CAPTIONING PROJECT
DATASET STRUCTURE DOCUMENTATION
==========================================

1. PROJECT ORGANIZATION

COCO_Project/
|
|-- annotations/
|   |-- instances_train2017.json
|   |-- captions_train2017.json
|
|-- images/
|   |-- knowledge_base/
|   |   |-- 10,000 selected images
|   |
|   |-- validation/
|       |-- 100 selected images
|
|-- metadata/
    |-- selected_image_ids.csv
    |-- images_without_object_annotations.csv
    |-- corrupted_images.csv
    |-- duplicate_images.csv
    |-- dataset_audit_report.json
    |-- dataset_structure.txt


2. DATASET CONFIGURATION

Dataset: MS COCO 2017
Image source: train2017
Random seed: 42

Knowledge Base: 10,000 images
Validation: 100 images
Total selected: 10,100 images


3. SOURCE ANNOTATIONS

Images in instances JSON: 118,287
Images in captions JSON: 118,287
Object annotations: 860,001
Caption annotations: 591,753
Object categories: 80


4. DATA INTEGRITY RESULTS

Missing images: 0
Corrupted images: 0
Exact duplicate groups: 0
Selected images missing from annotations: 0


5. ANNOTATION OBSERVATIONS

Images without object annotations: 104
Images without captions: 0

All 104 images without object annotations
have exactly five captions each.

Only 10 of these 104 images were manually inspected.


6. TEST DATASET

A separate test set of 100 images from
COCO val2017 is not included in this audit.
It must be prepared separately.


==========================================
END OF DOCUMENTATION
"""

structure_path = os.path.join(
    METADATA_DIR,
    "dataset_structure.txt"
)

with open(structure_path, "w") as file:
    file.write(structure_text.strip())

print("========== DATASET STRUCTURE DOCUMENTATION ==========")
print("File saved:", structure_path)
print("Documentation generated successfully!")

In [ ]:
# ==========================================
# 20. Verify Person 1 Deliverables
# ==========================================

required_files = [
    "selected_image_ids.csv",
    "images_without_object_annotations.csv",
    "corrupted_images.csv",
    "duplicate_images.csv",
    "dataset_audit_report.json",
    "dataset_structure.txt"
]

print("========== DELIVERABLES VERIFICATION ==========")

for filename in required_files:
    file_path = os.path.join(METADATA_DIR, filename)

    status = "FOUND" if os.path.exists(file_path) else "MISSING"

    print(f"{filename}: {status}")

# Verify JSON validity
with open(audit_report_path, "r") as file:
    json.load(file)

print("\nAudit JSON validity: PASSED")

# Final verification
all_exist = all(
    os.path.exists(os.path.join(METADATA_DIR, filename))
    for filename in required_files
)

print("All required files:", "PASSED" if all_exist else "FAILED")

## 21. Conclusion

The dataset exploration and data integrity audit of the selected MS COCO 2017 images was completed successfully.

A total of 10,100 images were selected from the train2017 dataset, consisting of 10,000 Knowledge Base images and 100 validation images.

The audit confirmed that all selected images are present, readable, and consistent with the source annotation files. No missing images, corrupted files, or exact duplicate images were detected.

The investigation also identified 104 images without object annotations. However, all 104 images have five associated captions each, and no selected images were found without captions.

All required audit reports and dataset documentation were generated and verified.

**Note:** A separate test set of 100 images from COCO val2017 remains to be prepared for the testing stage of the project.
